In [1]:
import numpy as np
import pandas as pd
from aiwhatif_cf.config import DATA_DIR, CF_OUTPUTS

batch_cfcheck_data = CF_OUTPUTS / "xgboost_random_2026-04-17" / "random_annotated_hltprhc.csv"
batch_cfcheck_df = pd.read_csv(batch_cfcheck_data)

In [2]:
pd.set_option("display.max_rows", None)

In [3]:
batch_cfcheck_df = batch_cfcheck_df.drop(columns=["hltprhc", "target_risk"])

In [4]:

batch_cfcheck_df["cf_id"] = batch_cfcheck_df["cf_id"].replace({"original": "xin"})

batch_cfcheck_df = batch_cfcheck_df.rename(columns={
    "original_risk": "risk_before",
    "predicted_risk" : "predicted_risk_after",
    "meets_target_risk" : "valid",
})

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.986591,0.0,0.025955,NaN,NaN,41.13
1,0,cf_1,NaN,NaN,5.0,5.0,NaN,NaN,NaN,NaN,0.025955,0.026657,False,NaN
2,0,cf_2,NaN,NaN,6.0,NaN,4.0,NaN,NaN,NaN,0.025955,0.027766,False,NaN
3,0,cf_3,NaN,NaN,NaN,6.0,NaN,NaN,21.184420,NaN,0.025955,0.063072,False,NaN


In [5]:
int_columns = [
    "etfruit",
    "eatveg",
    "cgtsmok",
    "alcfreq",
    "slprl",
    "paccnois",
    "dosprt",
]

float_columns = [
    "bmi",
    "risk_before",
    "predicted_risk_after",
]

In [6]:
batch_cfcheck_df[int_columns] = batch_cfcheck_df[int_columns].astype("Int64")

In [7]:
batch_cfcheck_df[float_columns] = batch_cfcheck_df[float_columns].round(2)

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,0.03,NaN,NaN,41.13
1,0,cf_1,<NA>,<NA>,5,5,<NA>,<NA>,NaN,<NA>,0.03,0.03,False,NaN
2,0,cf_2,<NA>,<NA>,6,<NA>,4,<NA>,NaN,<NA>,0.03,0.03,False,NaN
3,0,cf_3,<NA>,<NA>,<NA>,6,<NA>,<NA>,21.18,<NA>,0.03,0.06,False,NaN


In [8]:
batch_cfcheck_df[int_columns] = (
    batch_cfcheck_df[int_columns]
    .astype("string")
    .fillna("")
)

In [9]:
batch_cfcheck_df = batch_cfcheck_df.replace({np.nan : ""})
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,0.03,,,41.13
1,0,cf_1,,,5,5,,,,,0.03,0.03,False,
2,0,cf_2,,,6,,4,,,,0.03,0.03,False,
3,0,cf_3,,,,6,,,21.18,,0.03,0.06,False,


In [10]:
mask = batch_cfcheck_df["cf_id"] == "xin"
batch_cfcheck_df.loc[mask, "risk_before"] = ""
batch_cfcheck_df.head(4)

/tmp/ipykernel_38541/1160248817.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  batch_cfcheck_df.loc[mask, "risk_before"] = ""


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,,,,41.13
1,0,cf_1,,,5,5,,,,,0.03,0.03,False,
2,0,cf_2,,,6,,4,,,,0.03,0.03,False,
3,0,cf_3,,,,6,,,21.18,,0.03,0.06,False,


In [11]:
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# 1. Beräkna Nchanged endast för CF-rader
mask_cf = batch_cfcheck_df["cf_id"] != "xin"

batch_cfcheck_df.loc[mask_cf, "Nchanged"] = (
    batch_cfcheck_df.loc[mask_cf, feature_cols] != ""
).sum(axis=1)

# 2. Konvertera kolumnen till ren string
batch_cfcheck_df["Nchanged"] = batch_cfcheck_df["Nchanged"].astype("string")

# 3. Baseline ska vara tom
batch_cfcheck_df.loc[batch_cfcheck_df["cf_id"] == "xin", "Nchanged"] = ""

In [12]:
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec,Nchanged
0,0,xin,4,3,3,4,2,0,18.99,0,,,,41.13,
1,0,cf_1,,,5,5,,,,,0.03,0.03,False,,2
2,0,cf_2,,,6,,4,,,,0.03,0.03,False,,2
3,0,cf_3,,,,6,,,21.18,,0.03,0.06,False,,2


In [13]:
batch_cfcheck_df["Gower"] = ""
batch_cfcheck_df["Expected"] = ""


In [14]:
order = ["query_index", "cf_id"] + feature_cols + ["cf_gen_time_sec", "Gower", "Nchanged", "valid", "risk_before", "predicted_risk_after", "Expected"]

batch_cfcheck_df = batch_cfcheck_df[order]
batch_cfcheck_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,41.13,,,,,,
1,0,cf_1,,,5,5,,,,,,,2,False,0.03,0.03,
2,0,cf_2,,,6,,4,,,,,,2,False,0.03,0.03,
3,0,cf_3,,,,6,,,21.18,,,,2,False,0.03,0.06,
4,0,cf_4,,,1,,,,26.95,,,,2,False,0.03,0.04,
5,0,cf_5,,7,,,3,,,,,,2,False,0.03,0.02,
6,0,cf_6,,,,,,1,21.81,,,,2,False,0.03,0.02,
7,0,cf_7,7,,6,,,,,,,,2,True,0.03,0.0,
8,0,cf_8,6,2,,,,,,,,,2,True,0.03,0.01,
9,0,cf_9,,,,2,,,,,,,1,True,0.03,0.01,


# picking correct CF

### expected


In [15]:
# Check directional constraints directly
# These features should only improve (increase) or stay the same
SHOULD_INCREASE = ["cgtsmok", "alcfreq", "dosprt"]

# These features should only improve (decrease) or stay the same
SHOULD_DECREASE = ["bmi", "etfruit", "eatveg", "slprl", "paccnois"]


print("Directional constraints:")
print(f"  Should increase (↑): {SHOULD_INCREASE}")
print(f"  Should decrease (↓): {SHOULD_DECREASE}")


Directional constraints:
  Should increase (↑): ['cgtsmok', 'alcfreq', 'dosprt']
  Should decrease (↓): ['bmi', 'etfruit', 'eatveg', 'slprl', 'paccnois']


In [16]:
# Check if CFs violate directional constraints
# First, ensure numeric columns are actually numeric for comparison
numeric_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# Create a working copy with numeric values
df_check = batch_cfcheck_df.copy()
for col in numeric_cols:
    df_check[col] = pd.to_numeric(df_check[col], errors="coerce")

# Extract baseline values per query_index
baseline_values = df_check[df_check["cf_id"] == "xin"].set_index("query_index")[numeric_cols]

# Check violations for each CF
def check_violations(row):
    if row["cf_id"] == "xin":
        return ""

    baseline = baseline_values.loc[row["query_index"]]
    violations = []

    # Check features that should increase
    for feat in SHOULD_INCREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] < baseline[feat]:
                violations.append(f"{feat}↓")

    # Check features that should decrease
    for feat in SHOULD_DECREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] > baseline[feat]:
                violations.append(f"{feat}↑")

    return "NO: " + ", ".join(violations) if violations else ""

batch_cfcheck_df["Expected"] = df_check.apply(check_violations, axis=1)
batch_cfcheck_df.head(10)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,41.13,,,,,,
1,0,cf_1,,,5,5,,,,,,,2,False,0.03,0.03,
2,0,cf_2,,,6,,4,,,,,,2,False,0.03,0.03,NO: slprl↑
3,0,cf_3,,,,6,,,21.18,,,,2,False,0.03,0.06,NO: bmi↑
4,0,cf_4,,,1,,,,26.95,,,,2,False,0.03,0.04,"NO: cgtsmok↓, bmi↑"
5,0,cf_5,,7,,,3,,,,,,2,False,0.03,0.02,"NO: eatveg↑, slprl↑"
6,0,cf_6,,,,,,1,21.81,,,,2,False,0.03,0.02,"NO: bmi↑, paccnois↑"
7,0,cf_7,7,,6,,,,,,,,2,True,0.03,0.0,NO: etfruit↑
8,0,cf_8,6,2,,,,,,,,,2,True,0.03,0.01,NO: etfruit↑
9,0,cf_9,,,,2,,,,,,,1,True,0.03,0.01,NO: alcfreq↓


### is valid

In [17]:
# --- Select exactly one CF per query_index (random valid without violations) ---

# Split baseline and CF rows
df_xin = batch_cfcheck_df[batch_cfcheck_df["cf_id"] == "xin"]
df_cf  = batch_cfcheck_df[batch_cfcheck_df["cf_id"] != "xin"]

def select_random_cf(group):
    """Select one random CF that is valid and has no violations (Expected is empty)."""
    # First try: valid AND no violations
    good_cfs = group[(group["valid"] == True) & (group["Expected"] == "")]
    if len(good_cfs) > 0:
        return good_cfs.sample(n=1, random_state=42)

    # Fallback: any valid CF
    valid_cfs = group[group["valid"] == True]
    if len(valid_cfs) > 0:
        return valid_cfs.sample(n=1, random_state=42)

    # Last resort: first CF
    return group.iloc[[0]]

# Select one random CF per query_index
df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)

# Combine baseline + selected CF
batch_cfcheck_df = pd.concat([df_xin, df_cf_selected], ignore_index=True)

# Ensure xin appears before CF for each query_index
batch_cfcheck_df["is_xin"] = (batch_cfcheck_df["cf_id"] == "xin").astype(int)
batch_cfcheck_df = (
    batch_cfcheck_df
    .sort_values(["query_index", "is_xin"], ascending=[True, False])
    .drop(columns="is_xin")
)
batch_cfcheck_df

/tmp/ipykernel_38541/2325069138.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,41.13,,,,,,
9,0,cf_7,7,,6,,,,,,,,2,True,0.03,0.0,NO: etfruit↑
1,1,xin,3,4,1,2,3,0,22.38,0,0.16,,,,,,
10,1,cf_4,,,,4,,,,,,,1,True,0.25,0.08,
2,2,xin,5,3,1,1,4,0,22.69,7,0.16,,,,,,
11,2,cf_6,,,,,,,16.27,,,,1,True,0.22,0.01,
3,3,xin,3,3,6,6,2,0,24.34,1,48.81,,,,,,
12,3,cf_4,,1,,,,,,,,,1,True,0.07,0.0,
4,4,xin,5,4,2,7,1,0,21.26,3,42.86,,,,,,
13,4,cf_1,,,,3,,,,5,,,2,True,0.08,0.04,NO: alcfreq↓


In [18]:
batch_cfcheck_df["valid"] = batch_cfcheck_df["valid"].replace(
    {
        False : "No",
        True : "Yes",
    }
)